In [2]:
import joblib

In [3]:
mlp = joblib.load('mlp_url_model.pkl')
scaler = joblib.load('mlp_scaler.pkl')

In [4]:
import re
from urllib.parse import urlparse
from tld import get_tld

In [ ]:
def extract_features(url):
    parsed = urlparse(url)
    
    hostname = parsed.netloc
    path = parsed.path
    
    features = []

    # 1. hostname_length
    features.append(len(hostname))

    # 2. path_length
    features.append(len(path))

    # 3. fd_length (first directory length)
    try:
        features.append(len(path.split('/')[1]))
    except:
        features.append(0)

    # 4. tld_length
    try:
        tld = get_tld(url, fail_silently=True)
        features.append(len(tld) if tld else 0)
    except:
        features.append(0)

    # 5–10. caracteres especiales
    features.append(url.count('-'))   # count-
    features.append(url.count('@'))   # count@
    features.append(url.count('?'))   # count?
    features.append(url.count('%'))   # count%
    features.append(url.count('.'))   # count.
    features.append(url.count('='))   # count=

    # 11–13. palabras clave
    features.append(url.count('http'))   # count-http
    features.append(url.count('https'))  # count-https
    features.append(url.count('www'))    # count-www

    # 14. digits
    features.append(sum(c.isdigit() for c in url))

    # 15. letters
    features.append(sum(c.isalpha() for c in url))

    # 16. count_dir
    features.append(path.count('/'))

    # 17. use_of_ip
    ip_pattern = re.search(
        r'(\d{1,3}\.){3}\d{1,3}', url
    )
    features.append(1 if ip_pattern else 0)

    return [features]


In [19]:
def predict_url_final(url, threshold_block=90, threshold_review=60):
    features = extract_features(url)
    features_scaled = scaler.transform(features)

    probs = mlp.predict_proba(features_scaled)[0]
    benign_prob = probs[0] * 100
    malicious_prob = probs[1] * 100

    if malicious_prob >= threshold_block:
        decision = "Block URL"
        risk = "High"
        explanation = "High probability of malicious behavior detected"
    elif malicious_prob >= threshold_review:
        decision = "Review URL"
        risk = "Medium"
        explanation = "Suspicious patterns detected, manual review recommended"
    else:
        decision = "Allow URL"
        risk = "Low"
        explanation = "No significant malicious patterns detected"

    return {
        "url": url,
        "decision": decision,
        "risk_level": risk,
        "confidence": round(max(benign_prob, malicious_prob), 2),
        "malicious_probability": round(malicious_prob, 2),
        "benign_probability": round(benign_prob, 2),
        "explanation": explanation
    }


In [20]:
predict_url_final("https://www.google.com")

c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


{'url': 'https://www.google.com',
 'decision': 'Allow URL',
 'risk_level': 'Low',
 'confidence': np.float64(99.11),
 'malicious_probability': np.float64(0.89),
 'benign_probability': np.float64(99.11),
 'explanation': 'No significant malicious patterns detected'}

In [21]:
predict_url_final("https://login.miesbonos2025.com/login")

c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


{'url': 'https://login.miesbonos2025.com/login',
 'decision': 'Block URL',
 'risk_level': 'High',
 'confidence': np.float64(99.0),
 'malicious_probability': np.float64(99.0),
 'benign_probability': np.float64(1.0),
 'explanation': 'High probability of malicious behavior detected'}

In [22]:
predict_url_final("http://192.168.1.1/login/verify")

c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


{'url': 'http://192.168.1.1/login/verify',
 'decision': 'Block URL',
 'risk_level': 'High',
 'confidence': np.float64(100.0),
 'malicious_probability': np.float64(100.0),
 'benign_probability': np.float64(0.0),
 'explanation': 'High probability of malicious behavior detected'}